# Bias Magnitude Analysis

Quantifies how misleading standard offline evaluation is by measuring **bias magnitude** induced by the logging exposure mechanism.

- **Bias_M(π)** = M_biased(π) − M_unbiased(π) (Eq. 11)
- **Rating:** M ∈ {RMSE, MAE}; **Ranking:** M ∈ {NDCG@5, NDCG@10, Recall@5, Recall@10}
- **Unbiased sources:** Yahoo/Coat = randomized exposure; KuaiRec = near-complete matrix
- **Ranking distortion:** Kendall's τ(rank_biased, rank_unbiased)
- **Uncertainty:** 95% CI via bootstrap

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau

# Paths: set ROOT to this notebook's folder (bias_magnitude_analysis)
ROOT = os.getcwd()
BIASED_RESULTS_DIR = os.path.join(ROOT, "..", "biased data", "results")
BIASED_YAHOO = os.path.join(ROOT, "..", "biased data", "yahoo")
BIASED_COAT = os.path.join(ROOT, "..", "biased data", "coat shopping")
BIASED_KUAIREC = os.path.join(ROOT, "..", "biased data", "kuairec")
DATA_DIR = os.path.join(ROOT, "data")
UNBIASED_CSV = os.path.join(DATA_DIR, "unbiased_metrics.csv")
UNBIASED_TEMPLATE = os.path.join(DATA_DIR, "unbiased_metrics_template.csv")
ESS_CSV = os.path.join(DATA_DIR, "relative_ess_by_dataset.csv")
ESS_TEMPLATE = os.path.join(DATA_DIR, "relative_ess_template.csv")

## 1. Load biased metrics

From `biased data/results/metrics_*.txt` and `biased data/{yahoo,coat shopping,kuairec}/biased_results_*.csv`.

In [ ]:
def parse_metrics_txt(filepath):
    """Parse a metrics_*.txt file; return dict with RMSE, MAE and infer Dataset/Architecture from path."""
    with open(filepath, "r") as f:
        text = f.read()
    rmse = re.search(r"Test RMSE:\s*([\d.]+)", text)
    mae = re.search(r"Test MAE:\s*([\d.]+)", text)
    name = os.path.basename(filepath).lower()
    if "pranathi" in name and "mf" in name:
        dataset, arch = "YAHOO", "MF"
    elif "pranathi" in name:
        dataset, arch = "YAHOO", "NCF"
    elif "coat" in name:
        dataset, arch = "COAT", "NCF"
    elif "kuairec" in name and "mf" in name:
        dataset, arch = "KUAIREC", "MF"
    elif "kuairec" in name:
        dataset, arch = "KUAIREC", "NCF"
    else:
        dataset, arch = "UNKNOWN", "UNKNOWN"
    return {
        "Dataset": dataset,
        "Architecture": arch,
        "Model": "Naive ERM",
        "RMSE": float(rmse.group(1)) if rmse else np.nan,
        "MAE": float(mae.group(1)) if mae else np.nan,
        "NDCG@5": np.nan,
        "NDCG@10": np.nan,
        "Recall@5": np.nan,
        "Recall@10": np.nan,
    }


def load_biased_from_txt():
    rows = []
    if not os.path.isdir(BIASED_RESULTS_DIR):
        return pd.DataFrame(rows)
    for f in os.listdir(BIASED_RESULTS_DIR):
        if f.startswith("metrics_") and f.endswith(".txt"):
            path = os.path.join(BIASED_RESULTS_DIR, f)
            try:
                rows.append(parse_metrics_txt(path))
            except Exception as e:
                print(f"Skip {f}: {e}")
    return pd.DataFrame(rows)


def load_biased_from_csvs():
    """Optional legacy biased_results_*.csv loader (if present)."""
    dfs = []
    for folder in [BIASED_YAHOO, BIASED_COAT, BIASED_KUAIREC]:
        if not os.path.isdir(folder):
            continue
        for f in os.listdir(folder):
            if f.startswith("biased_results_") and f.endswith(".csv"):
                path = os.path.join(folder, f)
                try:
                    df = pd.read_csv(path)
                    if "Dataset" in df.columns and "Architecture" in df.columns:
                        dfs.append(df)
                except Exception as e:
                    print(f"Skip {path}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def infer_dataset_from_filename(name: str) -> str:
    n = name.lower()
    if "yahoo" in n or "pranathi" in n:
        return "YAHOO"
    if "coat" in n:
        return "COAT"
    if "kuairec" in n:
        return "KUAIREC"
    return "UNKNOWN"


def infer_arch_from_filename(name: str) -> str:
    n = name.lower()
    if "lightgcn" in n:
        return "LightGCN"
    if re.search(r"\bmf\b", n) or "_mf" in n:
        return "MF"
    if "ncf" in n:
        return "NCF"
    if "debiasing_results_yahoo.csv" in n or "debiasing_results_coat.csv" in n:
        return "NCF"
    return "UNKNOWN"


def load_biased_from_debiasing_results() -> pd.DataFrame:
    """Use existing `results/debiasing_results*.csv` as M_biased for ranking metrics.

    We take the `Naive ERM` row per file and attach Dataset/Architecture inferred
    from the filename. This is the only place in the repo where biased-side
    NDCG/Recall are already computed.
    """
    results_dir = os.path.join(ROOT, "..", "results")
    if not os.path.isdir(results_dir):
        return pd.DataFrame()

    dfs = []

    files = [f for f in os.listdir(results_dir) if f.startswith("debiasing_results") and f.endswith(".csv")]
    # Prefer more specific filenames first (e.g. *_lightgcn.csv, *_mf.csv) over generic ones.
    files = sorted(files, key=len, reverse=True)

    for f in files:
        path = os.path.join(results_dir, f)
        try:
            df = pd.read_csv(path)
        except Exception as e:
            print(f"Skip {path}: {e}")
            continue

        if "Model" not in df.columns:
            continue

        naive = df[df["Model"].astype(str).str.lower().str.contains("naive")].copy()
        if naive.empty:
            continue

        naive["Dataset"] = infer_dataset_from_filename(f)
        naive["Architecture"] = infer_arch_from_filename(f)
        naive["Model"] = "Naive ERM"

        # Skip unknowns (usually from generic files without a dataset/arch hint)
        naive = naive[(naive["Dataset"] != "UNKNOWN") & (naive["Architecture"] != "UNKNOWN")].copy()
        if naive.empty:
            continue

        keep_cols = [
            "Dataset",
            "Architecture",
            "Model",
            "RMSE",
            "MAE",
            "NDCG@5",
            "NDCG@10",
            "Recall@5",
            "Recall@10",
        ]
        for c in keep_cols:
            if c not in naive.columns:
                naive[c] = np.nan

        naive = naive[keep_cols]
        dfs.append(naive)

    out = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    if not out.empty:
        out = out.drop_duplicates(subset=["Dataset", "Architecture"], keep="first")
    return out


df_biased_deb = load_biased_from_debiasing_results()
df_biased_txt = load_biased_from_txt()
df_biased_csv = load_biased_from_csvs()

frames = [df for df in [df_biased_deb, df_biased_csv, df_biased_txt] if df is not None and not df.empty]
if frames:
    df_biased = pd.concat(frames, ignore_index=True)
    df_biased = df_biased.drop_duplicates(subset=["Dataset", "Architecture"], keep="first")
else:
    df_biased = pd.DataFrame(columns=["Dataset", "Architecture", "Model", "RMSE", "MAE", "NDCG@5", "NDCG@10", "Recall@5", "Recall@10"])

for c in ["RMSE", "MAE", "NDCG@5", "NDCG@10", "Recall@5", "Recall@10"]:
    if c not in df_biased.columns:
        df_biased[c] = np.nan

df_biased["Split"] = "biased"

out_biased_csv = os.path.join(DATA_DIR, "biased_metrics_from_debiasing.generated.csv")
df_biased.to_csv(out_biased_csv, index=False)
print("Wrote:", out_biased_csv)

print("Biased metrics (rows):", len(df_biased))
display(df_biased)

## 2. Load unbiased metrics

From `data/unbiased_metrics.csv`. If missing, use template and show instructions.

In [ ]:
METRIC_COLS_PRE = ["RMSE", "MAE", "NDCG@5", "NDCG@10", "Recall@5", "Recall@10"]
if os.path.isfile(UNBIASED_CSV):
    df_unbiased = pd.read_csv(UNBIASED_CSV)
    df_unbiased["Split"] = "unbiased"
else:
    if os.path.isfile(UNBIASED_TEMPLATE):
        df_unbiased = pd.read_csv(UNBIASED_TEMPLATE)
        df_unbiased["Split"] = "unbiased"
        print("Using unbiased template (fill values by evaluating on randomized/full-feedback sets).")
    else:
        df_unbiased = pd.DataFrame(columns=["Dataset", "Architecture", "Model"] + METRIC_COLS_PRE + ["Split"])
for c in METRIC_COLS_PRE:
    if c not in df_unbiased.columns:
        df_unbiased[c] = np.nan
print("Unbiased metrics (rows):", len(df_unbiased))
display(df_unbiased)

## 3. Compute bias magnitude

Bias_M(π) = M_biased(π) − M_unbiased(π). Report signed bias and |Bias_M|.

In [ ]:
METRIC_COLS = ["RMSE", "MAE", "NDCG@5", "NDCG@10", "Recall@5", "Recall@10"]
KEY_COLS = ["Dataset", "Architecture"]

def compute_bias_magnitude(df_b, df_u):
    if df_b.empty or df_u.empty:
        return pd.DataFrame()
    merged = df_b[KEY_COLS + METRIC_COLS].merge(
        df_u[KEY_COLS + METRIC_COLS],
        on=KEY_COLS,
        suffixes=("_biased", "_unbiased")
    )
    out = merged[KEY_COLS].copy()
    for m in METRIC_COLS:
        b = merged[f"{m}_biased"]
        u = merged[f"{m}_unbiased"]
        out[f"Bias_{m}"] = b - u
        out[f"|Bias_{m}|"] = np.abs(b - u)
    return out

# Normalize key columns so merge finds matches (e.g. strip whitespace)
for c in KEY_COLS:
    if c in df_biased.columns and df_biased[c].dtype == object:
        df_biased = df_biased.copy(); df_biased[c] = df_biased[c].astype(str).str.strip()
    if c in df_unbiased.columns and df_unbiased[c].dtype == object:
        df_unbiased = df_unbiased.copy(); df_unbiased[c] = df_unbiased[c].astype(str).str.strip()
df_bias = compute_bias_magnitude(df_biased, df_unbiased)
if df_bias.empty:
    print("No (Dataset, Architecture) pairs with both biased and unbiased metrics. Fill unbiased_metrics.csv and re-run.")
else:
    has_vals = df_bias[[c for c in df_bias.columns if c.startswith("Bias_")]].notna().any().any()
    if not has_vals:
        print("Bias magnitude (structure only — fill data/unbiased_metrics.csv with real unbiased metrics for actual bias):")
    else:
        print("Bias magnitude (signed and absolute):")
    display(df_bias)

## 4. Kendall's τ (ranking distortion)

τ = KendallTau(rank_biased, rank_unbiased). τ = 1 ⇒ identical rankings.

In [ ]:
def kendall_tau_for_metric(merged_df, metric, higher_better=True):
    """merged_df has columns metric_biased, metric_unbiased. Rank: higher is better by default."""
    b = merged_df[f"{metric}_biased"].values
    u = merged_df[f"{metric}_unbiased"].values
    valid = np.isfinite(b) & np.isfinite(u)
    if valid.sum() < 2:
        return np.nan, np.nan
    rank_b = pd.Series(b[valid]).rank(ascending=not higher_better)
    rank_u = pd.Series(u[valid]).rank(ascending=not higher_better)
    tau, pval = kendalltau(rank_b, rank_u)
    return tau, pval

if not df_bias.empty and len(df_bias) >= 2:
    merged = df_biased.merge(df_unbiased, on=KEY_COLS, suffixes=("_biased", "_unbiased"))
    print("Kendall's τ (ranking correlation, biased vs unbiased):")
    for m in ["NDCG@10", "Recall@10", "RMSE", "MAE"]:
        cb = f"{m}_biased" if f"{m}_biased" in merged.columns else None
        if cb and m in METRIC_COLS:
            higher = m in ["NDCG@5", "NDCG@10", "Recall@5", "Recall@10"]
            tau, p = kendall_tau_for_metric(merged, m, higher_better=higher)
            print(f"  {m}: τ = {tau:.4f}, p = {p:.4f}")
else:
    print("Need at least 2 model rows for Kendall's τ.")

## 5. Bootstrap 95% CI for bias

Bootstrap over (Dataset, Architecture) pairs when per-user data is unavailable.

In [ ]:
def bootstrap_bias_ci(df_bias, metric="Bias_RMSE", n_boot=2000, seed=42):
    if df_bias.empty or metric not in df_bias.columns:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    vals = df_bias[metric].dropna().values
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    boot_means = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    point = np.mean(vals)
    lo = np.percentile(boot_means, 2.5)
    hi = np.percentile(boot_means, 97.5)
    return point, lo, hi

if not df_bias.empty:
    print("Bootstrap 95% CI for mean bias (over models):")
    for m in METRIC_COLS:
        col = f"Bias_{m}"
        if col in df_bias.columns:
            point, lo, hi = bootstrap_bias_ci(df_bias, col)
            print(f"  {col}: {point:.4f} [{lo:.4f}, {hi:.4f}]")

## 6. Table 5 — Bias magnitude by dataset

Aggregated over model classes (MF, NCF, LightGCN).

In [ ]:
if df_bias.empty:
    print("Table 5: No bias data. Add unbiased_metrics.csv and re-run.")
else:
    bias_cols = [c for c in df_bias.columns if c.startswith("Bias_") or c.startswith("|Bias_")]
    if not bias_cols:
        print("Table 5: No bias columns.")
    else:
        agg = df_bias.groupby("Dataset")[bias_cols].mean().round(4)
        print("Table 5: Bias magnitude (mean over MF, NCF, LightGCN) by dataset")
        display(agg)
        agg.to_csv(os.path.join(DATA_DIR, "table5_bias_magnitude.csv"), index=True)

In [ ]:
# 7. Quick sanity plots inside this notebook (optional)
# Displays available generated PNGs inline.

from IPython.display import display, Image

root = ROOT  # bias_magnitude_analysis folder

candidates = [
    os.path.join(root, "popularity_quartiles_ndcg10.png"),
    os.path.join(root, "bias_ndcg10_vs_relative_ess.png"),
    os.path.join(root, "bias_rmse_vs_relative_ess.png"),
    os.path.join(root, "figure1_popularity_quartiles.generated.png"),
    os.path.join(root, "figure1_bias_vs_ess.generated.png"),
    os.path.join(root, "figure1_biasRMSE_vs_ess.generated.png"),
]

section7_dir = os.path.join(root, "..", "results", "section7")
section7_candidates = [
    os.path.join(section7_dir, "ips_gain_vs_relative_ess.png"),
    os.path.join(section7_dir, "ips_gain_vs_bias_ndcg10.png"),
    os.path.join(section7_dir, "section7_ips_gain_vs_ess.generated.png"),
    os.path.join(section7_dir, "section7_ips_gain_vs_bias.generated.png"),
]

print("Inline figures (existing files only):")

for path in candidates + section7_candidates:
    if os.path.isfile(path) and os.path.getsize(path) > 0:
        print("-", os.path.basename(path))
        display(Image(filename=path))

## 8. Bias vs relative ESS

Relates ranking-bias magnitude to estimator stability (relative ESS).

In [ ]:
ess_path = ESS_CSV if os.path.isfile(ESS_CSV) else ESS_TEMPLATE

if not df_bias.empty and os.path.isfile(ess_path):
    df_ess = pd.read_csv(ess_path)
    if "relative_ESS" in df_ess.columns:
        df_ess = df_ess.dropna(subset=["relative_ESS"])
    else:
        df_ess = pd.DataFrame()

    if not df_ess.empty and "|Bias_NDCG@10|" in df_bias.columns:
        plot_df = df_bias.merge(df_ess, on=["Dataset", "Architecture"], how="inner")
        plot_df = plot_df.dropna(subset=["relative_ESS", "|Bias_NDCG@10|"])

        if not plot_df.empty:
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.scatter(plot_df["relative_ESS"], plot_df["|Bias_NDCG@10|"])
            for _, row in plot_df.iterrows():
                ax.annotate(
                    f"{row['Dataset']}-{row['Architecture']}",
                    (row["relative_ESS"], row["|Bias_NDCG@10|"]),
                    xytext=(5, 5),
                    textcoords="offset points",
                    fontsize=8,
                )
            ax.set_xlabel("Relative ESS (ESS / N)")
            ax.set_ylabel("|Bias_NDCG@10|")
            ax.set_title("Bias magnitude vs relative ESS")
            plt.tight_layout()

            out_new = os.path.join(ROOT, "bias_ndcg10_vs_relative_ess.png")
            out_legacy = os.path.join(ROOT, "figure1_bias_vs_ess.png")
            plt.savefig(out_new, dpi=150)
            plt.savefig(out_legacy, dpi=150)
            plt.show()
            print("Saved:", out_new)
        else:
            print("No overlapping rows with valid relative_ESS and |Bias_NDCG@10|. Skipping plot.")
    else:
        print("Missing relative_ESS or |Bias_NDCG@10|. Skipping plot.")
else:
    print("Need both bias data and ESS file. Skipping plot (no placeholder generated).")